# Face Emotion Classifier - ViT (FER2013 to 6 Classes)
## Google Colab Version

Fine-tunes google/vit-base-patch16-224 on FER2013 with a 7-to-6 label mapping.

**Output classes:** Positive, Neutral, Stress, Anxiety, Negative, Depression

---
### Setup Instructions
1. **Runtime > Change runtime type > T4 GPU**
2. Run all cells in order (**Runtime > Run all**)
3. When prompted, authorize Google Drive access
4. Model saves to MyDrive/BE_models/face_final_6/ for persistence
---


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create output directory
DRIVE_PATH = '/content/drive/MyDrive/BE_models/face_final_6'
!mkdir -p {DRIVE_PATH}
print(f'Model will be saved to: {DRIVE_PATH}')


In [ ]:
!pip install -q transformers datasets evaluate scikit-learn matplotlib seaborn torchvision pillow


In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter
from datasets import load_dataset, Dataset, Image as HFImage
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
import evaluate


In [ ]:
# Load FER2013 dataset from HuggingFace
ds = load_dataset("abhilash88/fer2013-enhanced")
train_ds = ds["train"]
test_ds = ds["test"]
print(f"Train: {len(train_ds)}, Test: {len(test_ds)}")
print(f"Features: {train_ds.features}")
print(f"Label names: {train_ds.features["label"].names}")


In [ ]:
# FER2013 original 7 classes
# 0=Angry, 1=Disgust, 2=Fear, 3=Happy, 4=Sad, 5=Surprise, 6=Neutral
fer2013_labels = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"]

# Map FER2013 labels to BE 6 classes
# BE: 0=Positive, 1=Neutral, 2=Stress, 3=Anxiety, 4=Negative, 5=Depression
be_label_names = ["Positive", "Neutral", "Stress", "Anxiety", "Negative", "Depression"]

# Mapping: FER2013 → BE
# Angry(0) → Negative(4), Disgust(1) → Negative(4), Fear(2) → Anxiety(3)
# Happy(3) → Positive(0), Sad(4) → Depression(5), Surprise(5) → Positive(0), Neutral(6) → Neutral(1)
fer2be = {0: 4, 1: 4, 2: 3, 3: 0, 4: 5, 5: 0, 6: 1}

def convert_label(example):
    """Map FER2013 label to BE 6-class label."""
    return {"label": fer2be[example["label"]]}

train_ds = train_ds.map(convert_label)
test_ds = test_ds.map(convert_label)
print(f"Label distribution after mapping:")
train_counts = Counter(train_ds["label"])
for k in sorted(train_counts):
    print(f"  {be_label_names[k]}: {train_counts[k]}")


In [ ]:
# Class distribution in train set
train_labels = [ex["label"] for ex in train_ds]
test_labels = [ex["label"] for ex in test_ds]
print("Class distribution:")
print(f"{"Class":<15} {"Train":<10} {"Test":<10}")
print("-" * 35)
for i in range(6):
    train_c = train_labels.count(i)
    test_c = test_labels.count(i)
    print(f"{be_label_names[i]:<15} {train_c:<10} {test_c:<10}")

# Calculate class weights for imbalanced training
class_weights = []
max_count = max(train_counts.values())
for i in range(6):
    weight = max_count / train_counts[i]
    class_weights.append(weight)
print(f"
Class weights: {[round(w, 2) for w in class_weights]}")


In [ ]:
# Load ViT image processor
model_name = "google/vit-base-patch16-224"
image_processor = AutoImageProcessor.from_pretrained(model_name)
print(f"Image processor loaded: {type(image_processor).__name__}")
print(f"Image size: {image_processor.size}")


In [ ]:
# Define image transformation function
def transform_images(example):
    """Apply the image processor to convert PIL images to pixel values."""
    # example["image"] is a PIL Image
    inputs = image_processor(example["image"], return_tensors="pt")
    # Remove batch dimension from pixel_values
    example["pixel_values"] = inputs["pixel_values"][0]
    return example

# Apply transformation (this may take a minute)
train_ds = train_ds.map(transform_images, batched=False)
test_ds = test_ds.map(transform_images, batched=False)
print(f"Transformed train: {len(train_ds)}, test: {len(test_ds)}")
print(f"Sample pixel_values shape: {train_ds[0]["pixel_values"].shape}")

# Set format to keep only needed columns
keep_cols = ["pixel_values", "label"]
train_ds.set_format(type="torch", columns=keep_cols)
test_ds.set_format(type="torch", columns=keep_cols)
print("Format set to torch")


In [ ]:
# Load ViT model with 6 output classes
id2label = {i: name for i, name in enumerate(be_label_names)}
label2id = {name: i for i, name in enumerate(be_label_names)}

from transformers import ViTForImageClassification
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=6,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
print(f"Model: {model.config.model_type}")
print(f"Classes: {model.config.id2label}")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")


In [ ]:
acc = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = predictions.argmax(-1)
    accuracy = acc.compute(predictions=preds, references=labels)["accuracy"]
    f1_macro = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    f1_weighted = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }


In [ ]:
import torch
has_cuda = torch.cuda.is_available()
print(f"CUDA available: {has_cuda}")

output_dir = DRIVE_PATH  # saves to Google Drive

args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=20,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    remove_unused_columns=False,
    fp16=has_cuda,
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)
print(f"Training args configured: {args.num_train_epochs} epochs, bs={args.per_device_train_batch_size}, lr={args.learning_rate}")


In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)
print("Trainer initialized")


In [ ]:
trainer.train()

In [ ]:
# Evaluate the best model on the test set
test_results = trainer.evaluate()
print("Test Results:")
for key, value in test_results.items():
    print(f"  {key}: {value:.4f}")


In [ ]:
# Get detailed predictions for the test set
predictions = trainer.predict(test_ds)
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids
print(f"Predictions shape: {preds.shape}")
print(f"Labels shape: {labels.shape}")


In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

accuracy = accuracy_score(labels, preds)
f1_macro = f1_score(labels, preds, average="macro")
f1_weighted = f1_score(labels, preds, average="weighted")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 (macro): {f1_macro:.4f}")
print(f"F1 (weighted): {f1_weighted:.4f}")
print()
print("Classification Report:")
print(classification_report(labels, preds, target_names=be_label_names, digits=4))


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=be_label_names, yticklabels=be_label_names)
plt.title("Confusion Matrix - Face Model (FER2013 to 6 classes)")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig(f"{DRIVE_PATH}/face_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Confusion matrix saved to {DRIVE_PATH}/face_confusion_matrix.png")


In [ ]:
model.save_pretrained(DRIVE_PATH)
image_processor.save_pretrained(DRIVE_PATH)
print(f"Model saved to {DRIVE_PATH}")


In [ ]:
# Quick sanity check: load saved model and run inference on test images
from transformers import ViTForImageClassification, AutoImageProcessor
import torch
import numpy as np

# Load saved model and processor
saved_model = ViTForImageClassification.from_pretrained(DRIVE_PATH)
saved_processor = AutoImageProcessor.from_pretrained(DRIVE_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
saved_model.to(device)
saved_model.eval()
print("Saved model loaded successfully")

# Load a few test images from the dataset
test_samples = []
ds = load_dataset("abhilash88/fer2013-enhanced", split="test", streaming=True)
for i, example in enumerate(ds):
    if i >= 5:
        break
    test_samples.append(example)

# Run inference on each sample
be_labels = ["Positive", "Neutral", "Stress", "Anxiety", "Negative", "Depression"]
fer2013_labels = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"]
fer2be = {0: 4, 1: 4, 2: 3, 3: 0, 4: 5, 5: 0, 6: 1}
print("\nSanity Check Predictions:")
print("-" * 60)
for i, sample in enumerate(test_samples):
    inputs = saved_processor(sample["image"], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = saved_model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    pred_idx = outputs.logits.argmax(-1).item()
    orig_label = sample["label"]
    be_label = fer2be[orig_label]
    conf = probs[0][pred_idx].item()
    print(f"Sample {i+1}: Predicted={be_labels[pred_idx]:>10} (conf={conf:.3f}) | True(FER)={fer2013_labels[orig_label]:>8} -> BE={be_labels[be_label]:>10}")
print("\nSanity check complete!")


---
### Download model back to local project

**Option 1: Colab File Browser**
1. Click the **Files** icon in the left sidebar
2. Navigate to `drive/MyDrive/BE_models/face_final_6/`
3. Right-click the folder → **Download**
4. Unzip and place in your local `backend/models/face_final_6/`

**Option 2: Programmatic download** (run this cell locally after training):
```python
from google.colab import files
import zipfile, os, shutil, tempfile

# Zip the model folder and download
model_path = "/content/drive/MyDrive/BE_models/face_final_6"
with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as tmp:
    zip_path = tmp.name
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_ in os.walk(model_path):
        for fn in files_:
            fpath = os.path.join(root, fn)
            zf.write(fpath, os.path.relpath(fpath, os.path.dirname(model_path)))
files.download(zip_path)
os.unlink(zip_path)
```
---